<a href="https://colab.research.google.com/github/hauhauhau2006/IOT/blob/googlecolab/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import joblib # Thư viện dùng để xuất file .pkl bàn giao cho Thành viên 3

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/DATA_IOT/Selected dataset for ML and DL'))

['DNN-EdgeIIoT-dataset.csv', 'ML-EdgeIIoT-dataset.csv']


In [ ]:
# Đọc file dữ liệu đã được làm sạch
file_path = '/content/drive/MyDrive/DATA_IOT/Selected dataset for ML and DL/ML-EdgeIIoT-dataset.csv' # Sửa lại đường dẫn nếu cần
df = pd.read_csv(file_path, low_memory=False)

# Kiểm tra tổng quan dữ liệu
print(df.info())
print(df.head())

# Xử lý các cột non-numeric (object type) trong tập feature
# Lấy tất cả các cột có kiểu dữ liệu là 'object'
object_cols = df.select_dtypes(include='object').columns

# Loại bỏ các cột 'Attack_type' khỏi danh sách nếu nó là kiểu object (vì đây là target)
if 'Attack_type' in object_cols:
    object_cols = object_cols.drop('Attack_type')

# Tạo một bản sao của DataFrame để xử lý
df_processed = df.copy()

# Loại bỏ các cột object đã xác định từ df_processed
# NOTE: Nếu muốn sử dụng các cột này, cần phải thực hiện mã hóa (ví dụ: One-Hot Encoding, Label Encoding)
# Tuy nhiên, để khắc phục lỗi ValueError ngay lập tức, việc loại bỏ là giải pháp nhanh nhất.
df_processed = df_processed.drop(columns=object_cols)


# Tách Features (X) và Target (y)
# LƯU Ý: Thay 'Attack_type' bằng tên cột nhãn thực tế mà Vũ đã đặt trong file csv
X = df_processed.drop(columns=['Attack_type'])
y = df_processed['Attack_type']

# Phân chia dữ liệu thành tập huấn luyện (70%) và tập kiểm thử (30%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f"Kích thước tập train: {X_train.shape}")
print(f"Kích thước tập test: {X_test.shape}")

# 1. Danh sách các cột chứa text hoặc không đóng góp vào việc dự đoán cần loại bỏ
drop_columns = [
    "http_content_type", "http_file_data", "http_method", "http_uri",
    "arp_src_proto_ipv4", "arp_dst_proto_ipv4", "http_user_agent", "http_version",
    "dst_ip", "http_host", "src_ip", "tcp_options", "tcp_payload", "tcp_sport",
    "http_request_uri_query", "tcp_dsport", "source_file", "timestamp"
]

# 2. Xóa các cột này khỏi DataFrame
df = df.drop(columns=drop_columns, errors='ignore')

# 3. Xóa luôn các dòng bị rỗng (NaN) và trùng lặp để tránh nhiễu
df = df.dropna()
df = df.drop_duplicates()

# QUAN TRỌNG: Kiểm tra xem còn cột 'object' (text) nào lọt lưới không
print(df.select_dtypes(include=['object']).columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157800 entries, 0 to 157799
Data columns (total 63 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   frame.time                 157800 non-null  object 
 1   ip.src_host                157800 non-null  object 
 2   ip.dst_host                157800 non-null  object 
 3   arp.dst.proto_ipv4         157800 non-null  object 
 4   arp.opcode                 157800 non-null  float64
 5   arp.hw.size                157800 non-null  float64
 6   arp.src.proto_ipv4         157800 non-null  object 
 7   icmp.checksum              157800 non-null  float64
 8   icmp.seq_le                157800 non-null  float64
 9   icmp.transmit_timestamp    157800 non-null  float64
 10  icmp.unused                157800 non-null  float64
 11  http.file_data             157800 non-null  object 
 12  http.content_length        157800 non-null  float64
 13  http.request.uri.query     15

In [ ]:
def train_and_evaluate(model, model_name, X_train, y_train, X_test, y_test):
    print(f"--- Đang huấn luyện {model_name} ---")
    model.fit(X_train, y_train)

    # Dự đoán trên tập test
    y_pred = model.predict(X_test)

    # In các chỉ số đánh giá
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
    print(f"F1-score:  {f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\\n")
    return model

In [ ]:
# 1. Decision Tree
dt_model = DecisionTreeClassifier(random_state=42, max_depth=10)
trained_dt = train_and_evaluate(dt_model, "Decision Tree", X_train, y_train, X_test, y_test)

# 2. Random Forest (Có thể chạy hơi lâu tùy cấu hình Colab)
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
trained_rf = train_and_evaluate(rf_model, "Random Forest", X_train, y_train, X_test, y_test)

# 3. Support Vector Machine (Cần scale dữ liệu trước khi train SVM để chạy nhanh hơn)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm_model = SVC(random_state=42, kernel='linear')
trained_svm = train_and_evaluate(svm_model, "SVM", X_train_scaled, y_train, X_test_scaled, y_test)


--- Đang huấn luyện Decision Tree ---
Accuracy:  0.7562
Precision: 0.8032
Recall:    0.6318
F1-score:  0.6364
Confusion Matrix:
[[2752   39    0    0    0    0    0    0    0    0    7    0  261    0
     0]
 [   3  681    0    0    0    0    0    0   57    0    0  230 2195    1
     1]
 [   0    0 4210    0    0    0    0    0    0    0    0    0   17    0
     0]
 [   0    0    0 3073    0    0    0    0    0    0    0    0    1    0
     0]
 [   0    0    0    0 4349    0    0    0    0    0    0    0    0    0
     0]
 [   3    0    0    0    0    2    0    0    0    8    0    0  287    0
     0]
 [   0    0    0    0    0    0    0    0    0    0    0    0  364    0
     0]
 [   0    0    0    0    0    0    0 7290    0    0    0    0    0    0
     0]
 [   0  421    0    0    0    0    0    0 1075    0    0   13 1488    0
     0]
 [   0    0    0    0    0    0    0    0    0 2666    0    0  355    0
     0]
 [  88   67    0    0    0    0    0    0    0    0 2301    0  822    0


In [ ]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from google.colab import drive

# 1. Kết nối lại Google Drive (để đảm bảo không bị ngắt kết nối)
drive.mount('/content/drive', force_remount=True)

# 2. Đọc file dữ liệu đúng đường dẫn thực tế trên Drive của bạn
file_path = '/content/drive/MyDrive/DATA_IOT/Selected dataset for ML and DL/ML-EdgeIIoT-dataset.csv'
df = pd.read_csv(file_path, low_memory=False)

# 3. Mã hóa các cột dạng chuỗi (text) thành số để Random Forest có thể đọc được
object_cols = df.select_dtypes(include=['object']).columns
for col in object_cols:
    df[col] = df[col].astype(str)
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# 4. Tách Features (X) và Target (y) với đúng tên cột 'Attack_type'
X = df.drop(columns=['Attack_type'])
y = df['Attack_type']

# 5. Chia tập huấn luyện và kiểm thử
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# 6. Khởi tạo mô hình Random Forest
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=7,
    min_samples_split=30,
    min_samples_leaf=15,
    max_features='log2',
    max_leaf_nodes=40,
    ccp_alpha=0.01,
    bootstrap=True,
    random_state=42
)

# 7. Tiến hành huấn luyện
print("Đang huấn luyện mô hình Random Forest...")
rf_model.fit(X_train, y_train)

# 8. Xuất file .pkl ra Google Drive
best_model_path = '/content/drive/MyDrive/RandomForest_BestModel.pkl'
joblib.dump(rf_model, best_model_path)

print(f"✅ Đã xuất mô hình thành công và lưu tại: {best_model_path}")

Mounted at /content/drive
Đang huấn luyện mô hình Random Forest...
✅ Đã xuất mô hình thành công và lưu tại: /content/drive/MyDrive/RandomForest_BestModel.pkl
